# Dafne + MedSAM Thigh Segmentation

Runs the Dafne Thigh model slice-by-slice, then refines each muscle mask with MedSAM using the Dafne bounding box as the prompt. No manual point picking required.

MedSAM embedding is computed **once per slice** and reused for all 24 muscles.

**Kernel:** `dafne_clean`

In [1]:
import glob
import os
import sys
import numpy as np
import SimpleITK as sitk
import torch
from skimage import transform
from dafne_dl import DynamicDLModel
from dafne.config import GlobalConfig
from dafne.utils.sam_mask_refine import (
    load_sam,
    medsam_inference,
    enlarge_bounding_box,
    determine_device,
)

C:\Users\docto\miniconda3\envs\dafne_clean\lib\site-packages\requests\__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


In [8]:
# --- paths ---
MODALITY   = 'WATER'       # 'WATER' or 'FATFRACTION'

DEVICE     = determine_device()
MODEL_PATH = "dafne/extras/model/Thigh_1774532147.model"
IMAGE_GLOB = f"myosegmenTUM/*/ImageData/*{MODALITY}/*{MODALITY}_stack*.nii"
OUTPUT_DIR = f"dafne_medsam_results_{MODALITY.lower()}"

os.makedirs(OUTPUT_DIR, exist_ok=True)
print("Device  :", DEVICE)
print("Modality:", MODALITY)
print("Glob    :", IMAGE_GLOB)

SAM loaded on CPU
Device  : cpu
Modality: WATER
Glob    : myosegmenTUM/*/ImageData/*WATER/*WATER_stack*.nii


In [9]:
!pwd

/c/Projects/dissector/eval_notebooks


In [10]:
# load Dafne model
dafne_model = DynamicDLModel.Load(open(MODEL_PATH, "rb"))
print("Dafne model loaded:", MODEL_PATH)


Dafne model loaded: dafne/extras/model/Thigh_1774532147.model


In [11]:
# load MedSAM model once — downloads medsam_vit_b.pth (~375 MB) if not already present
GlobalConfig['SAM_MODEL'] = 'Med Sam'
sam_model = load_sam('Med Sam')
sam_model.eval()
print("MedSAM loaded on", DEVICE)

SAM loaded on CPU
MedSAM loaded on cpu


In [12]:
image_files = sorted(glob.glob(IMAGE_GLOB))
print(f"Found {len(image_files)} images:")
for p in image_files:
    print(" ", p)

Found 46 images:
  myosegmenTUM\HV001_1\ImageData\HV001_1_WATER\HV001_1_WATER_stack1.nii
  myosegmenTUM\HV001_1\ImageData\HV001_1_WATER\HV001_1_WATER_stack2.nii
  myosegmenTUM\HV001_2\ImageData\HV001_2_WATER\HV001_2_WATER_stack1.nii
  myosegmenTUM\HV001_2\ImageData\HV001_2_WATER\HV001_2_WATER_stack2.nii
  myosegmenTUM\HV001_3\ImageData\HV001_3_WATER\HV001_3_WATER_stack1.nii
  myosegmenTUM\HV001_3\ImageData\HV001_3_WATER\HV001_3_WATER_stack2.nii
  myosegmenTUM\HV002_1\ImageData\HV002_1_WATER\HV002_1_WATER_stack1.nii
  myosegmenTUM\HV002_1\ImageData\HV002_1_WATER\HV002_1_WATER_stack2.nii
  myosegmenTUM\HV002_2\ImageData\HV002_2_WATER\HV002_2_WATER_stack1.nii
  myosegmenTUM\HV002_2\ImageData\HV002_2_WATER\HV002_2_WATER_stack2.nii
  myosegmenTUM\HV002_3\ImageData\HV002_3_WATER\HV002_3_WATER_stack1.nii
  myosegmenTUM\HV002_3\ImageData\HV002_3_WATER\HV002_3_WATER_stack2.nii
  myosegmenTUM\HV003_1\ImageData\HV003_1_WATER\HV003_1_WATER_stack1.nii
  myosegmenTUM\HV003_1\ImageData\HV003_1_WATER\

In [ ]:
# run Dafne + MedSAM refinement slice-by-slice
for nii_path in image_files:
    stem     = os.path.splitext(os.path.basename(nii_path))[0]
    out_path = os.path.join(OUTPUT_DIR, f"{stem}_dafne_medsam.npz")

    if os.path.exists(out_path):
        print(f"Skipping (already done): {out_path}")
        continue

    print(f"\nProcessing: {nii_path}")
    img_sitk  = sitk.ReadImage(nii_path)
    img_array = sitk.GetArrayFromImage(img_sitk).astype(float)  # (slices, H, W)
    spacing   = img_sitk.GetSpacing()
    resolution = [spacing[0], spacing[1]]
    H, W = img_array.shape[1], img_array.shape[2]
    print(f"  Shape: {img_array.shape}  Resolution: {resolution}")

    all_masks = {}  # {muscle_name: 3D uint8 array}

    for slice_idx in range(img_array.shape[0]):
        slice_2d = img_array[slice_idx]

        # --- Dafne segmentation ---
        dafne_out = dafne_model({
            "image": slice_2d,
            "resolution": resolution,
            "split_laterality": True,
            "classification": "Thigh",
        })

        # --- MedSAM embedding (once per slice) ---
        img_norm = slice_2d * 255.0 / (slice_2d.max() + 1e-8)
        img_3c   = np.repeat(img_norm[:, :, None], 3, axis=-1)
        img_1024 = transform.resize(
            img_3c, (1024, 1024), order=3, preserve_range=True, anti_aliasing=True
        ).astype(np.uint8)
        img_1024 = (img_1024 - img_1024.min()) / np.clip(
            img_1024.max() - img_1024.min(), a_min=1e-8, a_max=None
        )
        img_tensor = torch.tensor(img_1024).float().permute(2, 0, 1).unsqueeze(0).to(DEVICE)
        with torch.no_grad():
            image_embedding = sam_model.image_encoder(img_tensor)

        # --- refine each muscle mask ---
        for muscle_name, mask in dafne_out.items():
            mask_arr = np.asarray(mask, dtype=np.uint8)

            if mask_arr.any():
                bbox     = enlarge_bounding_box(mask_arr)              # [min_col, min_row, max_col, max_row]
                box_1024 = bbox / np.array([W, H, W, H]) * 1024
                box_1024 = box_1024[None, None, :]                     # (1, 1, 4)
                refined  = medsam_inference(sam_model, image_embedding, box_1024, H, W)
            else:
                refined = mask_arr

            if muscle_name not in all_masks:
                all_masks[muscle_name] = np.zeros(img_array.shape, dtype=np.uint8)
            all_masks[muscle_name][slice_idx] = refined.astype(np.uint8)

        if (slice_idx + 1) % 5 == 0 or slice_idx == img_array.shape[0] - 1:
            print(f"  slice {slice_idx + 1}/{img_array.shape[0]} done")

    np.savez_compressed(out_path, **all_masks)
    print(f"  Saved → {out_path}")
    print(f"  Muscles: {list(all_masks.keys())}")

print("\nAll done.")


Processing: myosegmenTUM\HV001_1\ImageData\HV001_1_WATER\HV001_1_WATER_stack1.nii
  Shape: (65, 672, 672)  Resolution: [1.0, 1.0]
Biascorrection in model. Image Max: 346.7769
1/1 ━━━━━━━━━━━━━━━━━━━━ 4s 4s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 745ms/step


In [ ]:
# sanity check — reload one result
results = sorted(glob.glob(os.path.join(OUTPUT_DIR, "*.npz")))
if results:
    sample = np.load(results[0])
    print("Sample file:", results[0])
    for name in sample.files:
        arr = sample[name]
        print(f"  {name}: shape={arr.shape}  positive voxels={arr.sum()}")